# NB08 — Acoustic Signal-Chain Characterization & TDOA (Pico Sweep)

## Signal path
```
Pico GP15 (Pin 20) → 100 Ω → Piezo transmitter
Pico GP14 (Pin 19) → Rigol CH1 trigger/marker
                                    ↓ acoustic / mechanical coupling
                        CH3 (Stahl + LM358 + 830 kΩ, calibrated probe head A) → Rigol CH3
                        CH4 (Stahl + LM358 + 830 kΩ, calibrated probe head B) → Rigol CH4
```

Probe-head status: calibrated on 2026-06-29.

**CH1 is timing only — it is not an acoustic sensor in the active Pico sweep setup.**
Sweep windows are identified from the GP14 marker pulses; CH3 and CH4 carry the acoustic response.

## What this notebook measures
1. **Transfer function** — amplitude vs. frequency for CH3 and CH4 (normalized to peak)
2. **LM358 bandwidth** — −3 dB point from CH3 and CH4 curves
3. **Phase-slope TDE** — Δφ vs ω linear fit for CH3−CH4 → propagation delay τ
4. **TDOA geometry** — limited to the CH3/CH4 delay unless a third calibrated acoustic channel is added

## Prerequisites
- Pico running the Rust PIO sweep firmware from `test_signals/pico_rust_sweep`
- GP14 connected to Rigol CH1 and selected as trigger source
- NB04 stopped: `./measurement.py stop`
- CH3 and CH4 connected to calibrated Stahl+LM358+830 kΩ probe heads

## Pico sweep
- 20 Hz → 100 kHz, linear, 5 s per sweep
- GP14 marker pulse at sweep start and every 5 kHz window
- 20 windows × 5 kHz, about 250 ms per window

## References
- De Roo et al. (2016) — AE propagation in plant tissue
- Carter (1987) — GCC-PHAT TDE
- Khait et al. (2023) — plant AE 20–150 kHz


In [ ]:
%matplotlib inline
import sys, time, json, math
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import differential_evolution
from scipy import signal as sp_signal

CHAR_ROOT = Path('..').resolve()
sys.path.insert(0, str(CHAR_ROOT))

from scope.config import load_config
from scope.instrument import InstrumentConnection, acquire_waveform_bytes, get_preamble
from plant_ae.watering import CONFIG_PATH

# ── Pico sweep parameters (must match test_signals/pico_rust_sweep) ─────────
PICO_F_START_HZ      = 20
PICO_F_STOP_HZ       = 100_000
PICO_SWEEP_S         = 5.0
PICO_MARKER_STEP_HZ  = 5_000
PICO_WINDOW_S        = PICO_SWEEP_S * PICO_MARKER_STEP_HZ / (PICO_F_STOP_HZ - PICO_F_START_HZ)
PICO_FREQUENCIES     = list(range(PICO_MARKER_STEP_HZ, PICO_F_STOP_HZ + 1, PICO_MARKER_STEP_HZ))
N_SWEEPS             = 3

# ── Scope ─────────────────────────────────────────────────────────────────────
SAMPLE_RATE_HZ = 500_000
MEMORY_DEPTH   = 50_000     # 100 ms window, Δf = 10 Hz
CHANNELS       = (1, 3, 4)  # CH1 marker; CH3/CH4 acoustic response
SENSOR_CHANNELS = (3, 4)

# CH1: Pico GP14 marker/trigger          → 1 V/div
# CH3: Stahl + LM358 + 830kΩ, calibrated → 500 mV/div
# CH4: Stahl + LM358 + 830kΩ, calibrated → 500 mV/div
CH_SCALE = {1: 1.0, 3: 0.5, 4: 0.5}
CH_PROBE = {1: 1, 3: 10, 4: 10}

FREQ_LO, FREQ_HI = 20, 100_000

# ── Output ────────────────────────────────────────────────────────────────────
RUN_ID  = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = CHAR_ROOT / 'data' / 'pico_sweep_tde' / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Run    : {RUN_ID}')
print(f'Output : {OUT_DIR}')
print(f'Pico   : {len(PICO_FREQUENCIES)} marker windows × {PICO_WINDOW_S:.3f}s = {len(PICO_FREQUENCIES)*PICO_WINDOW_S/60:.1f} min/sweep')
print(f'Scope  : {SAMPLE_RATE_HZ//1000} kSa/s, {MEMORY_DEPTH} pts, Δf={SAMPLE_RATE_HZ/MEMORY_DEPTH:.0f} Hz')
print(f'Plan   : {N_SWEEPS} sweeps = {N_SWEEPS * len(PICO_FREQUENCIES) * PICO_WINDOW_S / 60:.0f} min total')


In [ ]:
# ── Scope helpers ─────────────────────────────────────────────────────────────
config = load_config(CONFIG_PATH)

def configure_sweep_scope(conn):
    conn.write(':STOP'); time.sleep(0.2)
    for ch in CHANNELS:
        conn.write(f':CHAN{ch}:DISP ON')
        conn.write(f':CHAN{ch}:COUP AC')
        conn.write(f':CHAN{ch}:PROB {CH_PROBE[ch]}')
        conn.write(f':CHAN{ch}:SCAL {CH_SCALE[ch]}')
        conn.write(f':CHAN{ch}:OFFS 0')
    conn.write(':CHAN2:DISP OFF')  # CH2 defekt
    conn.write(f':ACQ:MDEP {MEMORY_DEPTH}')
    conn.write(f':ACQ:SRAT {SAMPLE_RATE_HZ}')
    conn.write(':ACQ:TYPE NORM')
    timebase = MEMORY_DEPTH / (12 * SAMPLE_RATE_HZ)
    conn.write(f':TIM:SCAL {timebase}')
    conn.write(':TIM:OFFS 0')
    conn.write(':TRIG:MODE EDGE')
    conn.write(':TRIG:EDGE:SOUR CHAN1')  # CH1: Pico GP14 marker
    conn.write(':TRIG:EDGE:LEV 1.0')
    conn.write(':TRIG:SWE NORM')
    conn.write(':RUN'); time.sleep(1.0)
    rate = float(conn.query(':ACQ:SRAT?'))
    depth = int(float(conn.query(':ACQ:MDEP?')))
    print(f'  Scope configured: {rate/1e3:.0f} kSa/s, {depth} pts')

def acquire_frame(conn):
    conn.write(':SING')
    deadline = time.monotonic() + 5.0
    while time.monotonic() < deadline:
        if conn.query(':TRIG:STAT?').strip().upper() == 'STOP':
            break
        time.sleep(0.05)
    voltages = {}
    for ch in CHANNELS:
        pre, raw = acquire_waveform_bytes(conn, ch, mode='RAW', fmt='BYTE',
                                          max_points=MEMORY_DEPTH)
        v = (np.frombuffer(bytes(raw), dtype=np.uint8).astype(np.float32)
             - pre.yorigin - pre.yreference) * pre.yincrement
        voltages[ch] = v
    return voltages

def detect_pico_freq(voltages):
    """Detect drive frequency from CH3 FFT peak (strongest amplified signal)."""
    sig   = voltages[3]
    n     = len(sig)
    freqs = np.fft.rfftfreq(n, 1 / SAMPLE_RATE_HZ)
    spec  = np.abs(np.fft.rfft(sig * np.hanning(n)))
    mask  = (freqs >= FREQ_LO) & (freqs <= FREQ_HI)
    idx   = np.argmax(spec[mask])
    return float(freqs[mask][idx])

def fft_amplitude(sig, freq):
    n     = len(sig)
    freqs = np.fft.rfftfreq(n, 1 / SAMPLE_RATE_HZ)
    spec  = np.abs(np.fft.rfft(sig * np.hanning(n))) * 2 / n
    return float(spec[np.argmin(np.abs(freqs - freq))])

def fft_phase(sig, freq):
    n     = len(sig)
    freqs = np.fft.rfftfreq(n, 1 / SAMPLE_RATE_HZ)
    spec  = np.fft.rfft(sig * np.hanning(n))
    return float(np.angle(spec[np.argmin(np.abs(freqs - freq))]))

print('Helpers ready.')


## Part 1 — Acquisition

In [ ]:
records = []
TARGET  = N_SWEEPS * len(PICO_FREQUENCIES)

print(f'Collecting {TARGET} frames — press Ctrl-C to stop early\n')

try:
    with InstrumentConnection(config) as conn:
        configure_sweep_scope(conn)
        while len(records) < TARGET:
            try:
                v    = acquire_frame(conn)
                det  = detect_pico_freq(v)
                near = PICO_FREQUENCIES[len(records) % len(PICO_FREQUENCIES)]
                rec  = {
                    'frame': len(records) + 1,
                    'ts': datetime.now().isoformat(),
                    'detected_hz': det,
                    'nearest_pico_hz': near,
                    'freq_error_hz': abs(det - near),
                    'marker_window_hz': near,
                }
                for ch in CHANNELS:
                    rec[f'amp_ch{ch}']   = fft_amplitude(v[ch], det)
                    rec[f'phase_ch{ch}'] = fft_phase(v[ch], det)
                records.append(rec)
                print(f"  [{rec['frame']:3d}/{TARGET}] {det:8.0f} Hz → {near} Hz  "
                      f"CH1={20*np.log10(rec['amp_ch1']+1e-12):+5.1f}  "
                      f"CH3={20*np.log10(rec['amp_ch3']+1e-12):+5.1f}  "
                      f"CH4={20*np.log10(rec['amp_ch4']+1e-12):+5.1f} dBV",
                      flush=True)
            except Exception as e:
                print(f'  frame error: {e}', flush=True)
                time.sleep(1)
except KeyboardInterrupt:
    print('\nStopped early.')

with (OUT_DIR / 'records.jsonl').open('w') as f:
    for r in records: f.write(json.dumps(r) + '\n')
print(f'\n{len(records)} frames → {OUT_DIR}/records.jsonl')


## Part 2 — Transfer function (amplitude vs frequency)

In [ ]:
# Group by Pico step, average amplitude across sweeps
by_freq = defaultdict(list)
for r in records:
    by_freq[r.get('marker_window_hz', r['nearest_pico_hz'])].append(r)

freqs  = sorted(by_freq)
amp    = {ch: [] for ch in SENSOR_CHANNELS}   # V RMS average per frequency
n_avg  = []

for f in freqs:
    recs = by_freq[f]
    n_avg.append(len(recs))
    for ch in SENSOR_CHANNELS:
        amp[ch].append(np.mean([r[f'amp_ch{ch}'] for r in recs]))

# Normalize each channel to its own peak → show shape only
def to_db(arr):
    a = np.array(arr, dtype=float)
    peak = max(a)
    return 20 * np.log10(np.maximum(a, 1e-12) / max(peak, 1e-12))

db = {ch: to_db(amp[ch]) for ch in CHANNELS}

# Find −3 dB frequency for CH3 and CH4
def find_3db(freqs, db_arr):
    for i in range(1, len(freqs)):
        if db_arr[i] < -3.0:
            return freqs[i]
    return None

f3db = {ch: find_3db(freqs, db[ch]) for ch in SENSOR_CHANNELS}
for ch in SENSOR_CHANNELS:
    print(f'CH{ch} −3dB: {f3db[ch]} Hz' if f3db[ch] else f'CH{ch} −3dB: not reached')

# Plot
CH_COLOR = {3: 'cyan', 4: 'magenta'}
CH_LABEL = {3: 'CH3 steel+LM358+830kΩ calibrated', 4: 'CH4 steel+LM358+830kΩ calibrated'}

fig, ax = plt.subplots(figsize=(12, 5))
for ch in SENSOR_CHANNELS:
    ax.semilogx(freqs, db[ch], '-o', ms=4, color=CH_COLOR[ch], label=CH_LABEL[ch])
    if f3db[ch]:
        ax.axvline(f3db[ch], color=CH_COLOR[ch], ls='--', lw=0.9,
                   label=f'CH{ch} −3dB @ {f3db[ch]/1e3:.1f} kHz')
ax.axhline(-3, color='k', ls=':', lw=0.8, label='−3 dB')
ax.axvspan(20_000, 150_000, alpha=0.07, color='yellow', label='AE range (Khait 2023)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Normalized amplitude (dB)')
ax.set_title(f'Transfer function — Pico sweep marker windows {RUN_ID}')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'transfer_function.png', dpi=150)
plt.show()


## Part 3 — Phase-slope TDE (all 3 sensor pairs)

In [ ]:
PAIRS = [
    (3, 4, 'CH3−CH4'),
]

# Average phase per (pair, frequency)
ph_mean = {}
for a, b, lbl in PAIRS:
    diffs = []
    for f in freqs:
        recs = by_freq[f]
        d = np.mean([r[f'phase_ch{a}'] - r[f'phase_ch{b}'] for r in recs])
        diffs.append(d)
    ph_mean[(a, b)] = np.unwrap(diffs)

# Phase-slope TDE: τ = dΔφ/dω
omegas = np.array([2 * np.pi * f for f in freqs])
tde_ps = {}   # τ in ms

fig, axes = plt.subplots(len(PAIRS), 1, figsize=(12, 4), sharex=True)
axes = np.atleast_1d(axes)
fig.suptitle(f'Phase-slope TDE — {RUN_ID}', fontsize=11)

for ax, (a, b, lbl) in zip(axes, PAIRS):
    ph = ph_mean[(a, b)]
    # Use only bands where both channels have decent amplitude (>−20dB)
    valid = (np.array(db[a]) > -20) & (np.array(db[b]) > -20)
    tau_s = None
    if valid.sum() >= 3:
        coeffs  = np.polyfit(omegas[valid], ph[valid], 1)
        tau_s   = coeffs[0]
        tau_us  = tau_s * 1e6
        fit_deg = np.degrees(np.polyval(coeffs, omegas))
        ax.semilogx(freqs, fit_deg, 'r--', lw=1.5,
                    label=f'Linear fit  τ = {tau_us:+.2f} µs')
    tde_ps[(a, b)] = tau_s
    ax.semilogx(freqs, np.degrees(ph), 'o-', ms=4, color='steelblue',
                label=f'Δφ {lbl} (unwrapped)')
    ax.axvspan(20_000, 150_000, alpha=0.07, color='yellow')
    ax.set_ylabel('Phase diff (°)'); ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))

axes[-1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'phase_tde.png', dpi=150)
plt.show()

print('\n=== Phase-slope TDE results ===')
for a, b, lbl in PAIRS:
    tau_s = tde_ps.get((a, b))
    if tau_s is not None:
        tau_us = tau_s * 1e6
        print(f'{lbl}: τ = {tau_us:+.2f} µs')
        for medium, v in [('steel path', 5000), ('wet substrate', 400), ('dry substrate', 200)]:
            print(f'   {medium:13s}  d = {abs(tau_s)*v*100:.1f} cm  (v={v} m/s)')
    else:
        print(f'{lbl}: not enough valid points')


## Part 4 — Geometry note

With the active 2026-06-29 wiring, only CH3 and CH4 are calibrated acoustic response channels.
CH1 is the GP14 marker/trigger and must not be used as an acoustic sensor.

A full 2D TDOA geometry fit needs at least three acoustic sensors. This notebook therefore reports
only the CH3−CH4 phase-slope delay unless a third calibrated acoustic channel is added.


In [ ]:
# ── Observed TDE in seconds ───────────────────────────────────────────────────
tau_obs = {pair: tde_ps[pair] for pair in tde_ps if tde_ps[pair] is not None}

if (3, 4) in tau_obs:
    tau_us = tau_obs[(3, 4)] * 1e6
    print(f'CH3−CH4 delay: {tau_us:+.2f} µs')
    print('Full TDOA geometry skipped: CH1 is the GP14 marker, not an acoustic sensor.')
else:
    print('No valid CH3−CH4 TDE available.')
